In [ ]:
# ============================================================================
# WEIGHT SENSITIVITY ANALYSIS — Standalone cell (response to W4)
#
# The unsupervised ensemble in §5.3 uses ten heuristic weights:
#   LLM block:    w_llama, w_qwen, w_mistral, w_qwen32b   (each 0.25)
#                 w_inter2, w_inter3, w_inter4            (0.3, 0.5, 1.0)
#   Lexical:      w_tfidf, w_bm25, w_crossencoder         (0.2, 0.2, 0.4)
#
# This cell tests robustness to those choices via three ablations:
#   (1) LLM-vs-lexical balance sweep  — direct response to the reviewer
#   (2) Random perturbation           — multiplies each weight by U[0.5, 1.5]
#   (3) Component dropout             — sets one weight to 0 at a time
#
# Self-contained: tries to reuse `df` from a previous main() call,
# otherwise loads the most recent df_with_scores_*.csv.
# ============================================================================

import os, glob, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# 1) Configuration
# ---------------------------------------------------------------------------
WS_BASE_PATH   = "artifacts"  # not shipped — see DATA.md
WS_OUTPUT_DIR  = f"{WS_BASE_PATH}/outputs_unsupervised_4llm_all"
WS_RESULTS_DIR = os.path.join(WS_OUTPUT_DIR, "weight_sensitivity")
os.makedirs(WS_RESULTS_DIR, exist_ok=True)

WS_K = 200       # focal cutoff for the rebuttal
WS_N_PERTURB = 200
WS_SEED = 42

# Paper weights (from §5.3)
WS_W_PAPER = {
    "vote_llama":   0.25, "vote_qwen":    0.25,
    "vote_mistral": 0.25, "vote_qwen32b": 0.25,
    "vote_inter2":  0.30, "vote_inter3":  0.50, "vote_inter4": 1.00,
    "sim_tfidf":    0.20, "sim_bm25":     0.20, "sim_crossencoder": 0.40,
}
WS_LLM_KEYS = ["vote_llama", "vote_qwen", "vote_mistral", "vote_qwen32b",
               "vote_inter2", "vote_inter3", "vote_inter4"]
WS_LEX_KEYS = ["sim_tfidf", "sim_bm25", "sim_crossencoder"]

# ---------------------------------------------------------------------------
# 2) Get the dataframe (from scope or from CSV)
# ---------------------------------------------------------------------------
try:
    ws_df_full = df.copy()
    print(f"Reusing `df` from scope (n = {len(ws_df_full)})")
except NameError:
    pattern = os.path.join(WS_OUTPUT_DIR, "df_with_scores_*.csv")
    ws_matches = sorted(glob.glob(pattern))
    if not ws_matches:
        raise FileNotFoundError(
            f"No saved df at {pattern}. Run main() first, or check WS_OUTPUT_DIR."
        )
    ws_df_full = pd.read_csv(ws_matches[-1])
    print(f"Loaded {ws_matches[-1]} (n = {len(ws_df_full)})")

# Sanity check
ws_required = WS_LLM_KEYS + WS_LEX_KEYS + ["lbl_Gold"]
ws_missing = [c for c in ws_required if c not in ws_df_full.columns]
if ws_missing:
    raise ValueError(f"Missing columns: {ws_missing}")

# Restrict to rows with a Gold label (= benchmark of 1015 pairs)
ws_mask = ws_df_full["lbl_Gold"].notna()
ws_df = ws_df_full[ws_mask].reset_index(drop=True).copy()
ws_y = ws_df["lbl_Gold"].values.astype(int)
ws_n_pos = int(ws_y.sum())
print(f"Gold subset: n = {len(ws_df)}, positives = {ws_n_pos}, "
      f"prevalence = {ws_n_pos/len(ws_df):.3f}")

# ---------------------------------------------------------------------------
# 3) Helpers: score from weights, AP, P@k, R@k
# ---------------------------------------------------------------------------
def ws_compute_score(df_sub, weights):
    s = np.zeros(len(df_sub))
    for k, w in weights.items():
        if k in df_sub.columns and w != 0:
            s += w * df_sub[k].values
    return s

def ws_ap(y, scores):
    if len(y) == 0 or y.sum() == 0:
        return 0.0
    order = np.argsort(scores)[::-1]
    y_s = y[order]
    cum = np.cumsum(y_s)
    precs = cum / np.arange(1, len(y_s) + 1)
    return float((precs * y_s).sum() / y_s.sum())

def ws_pk(y, scores, k):
    k = min(k, len(y))
    return float(y[np.argsort(scores)[::-1]][:k].mean())

def ws_rk(y, scores, k):
    if y.sum() == 0: return 0.0
    k = min(k, len(y))
    return float(y[np.argsort(scores)[::-1]][:k].sum() / y.sum())

# Reference: paper config
ws_score_paper = ws_compute_score(ws_df, WS_W_PAPER)
ws_ap_paper  = ws_ap(ws_y, ws_score_paper)
ws_p_paper   = ws_pk(ws_y, ws_score_paper, WS_K)
ws_r_paper   = ws_rk(ws_y, ws_score_paper, WS_K)
print(f"\nPaper config: AP = {ws_ap_paper:.3f}, "
      f"P@{WS_K} = {ws_p_paper:.3f}, R@{WS_K} = {ws_r_paper:.3f}")

# ---------------------------------------------------------------------------
# 4) Ablation 1 — LLM-vs-lexical balance sweep
#    Multiply the LLM block by a factor λ ∈ {0, 0.1, ..., ∞}
#    Lexical block stays at paper values.
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print(f"ABLATION 1 — λ sweep (scales the LLM block; lexical block unchanged)")
print("=" * 70)

ws_lambdas = [0.0, 0.1, 0.25, 0.5, 1.0, 2.0, 4.0, 10.0, np.inf]
ws_sweep = []
for lam in ws_lambdas:
    if np.isinf(lam):
        w = {k: (WS_W_PAPER[k] if k in WS_LLM_KEYS else 0.0) for k in WS_W_PAPER}
        label = "∞ (LLM-only)"
    elif lam == 0:
        w = {k: (0.0 if k in WS_LLM_KEYS else WS_W_PAPER[k]) for k in WS_W_PAPER}
        label = "0 (lexical-only)"
    else:
        w = {k: (WS_W_PAPER[k] * lam if k in WS_LLM_KEYS else WS_W_PAPER[k])
             for k in WS_W_PAPER}
        label = f"{lam:g}{' (paper)' if lam == 1.0 else ''}"
    s = ws_compute_score(ws_df, w)
    ws_sweep.append({"λ_LLM": label, "AP": ws_ap(ws_y, s),
                     f"P@{WS_K}": ws_pk(ws_y, s, WS_K),
                     f"R@{WS_K}": ws_rk(ws_y, s, WS_K)})
ws_sweep_df = pd.DataFrame(ws_sweep)
print(ws_sweep_df.round(3).to_string(index=False))
ws_sweep_df.to_csv(os.path.join(WS_RESULTS_DIR, "ablation1_lambda_sweep.csv"), index=False)

# ---------------------------------------------------------------------------
# 5) Ablation 2 — Random perturbation: each weight × U[0.5, 1.5]
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print(f"ABLATION 2 — Random perturbation (each weight × U[0.5, 1.5], "
      f"N = {WS_N_PERTURB})")
print("=" * 70)

ws_rng = np.random.default_rng(WS_SEED)
ws_perturb = []
for _ in range(WS_N_PERTURB):
    w = {k: WS_W_PAPER[k] * ws_rng.uniform(0.5, 1.5) for k in WS_W_PAPER}
    s = ws_compute_score(ws_df, w)
    ws_perturb.append({"AP": ws_ap(ws_y, s),
                       f"P@{WS_K}": ws_pk(ws_y, s, WS_K),
                       f"R@{WS_K}": ws_rk(ws_y, s, WS_K)})
ws_perturb_df = pd.DataFrame(ws_perturb)

for col in ["AP", f"P@{WS_K}", f"R@{WS_K}"]:
    v = ws_perturb_df[col]
    print(f"  {col:<8}: mean={v.mean():.3f}  std={v.std():.3f}  "
          f"min={v.min():.3f}  max={v.max():.3f}  "
          f"q5={v.quantile(0.05):.3f}  q95={v.quantile(0.95):.3f}")
ws_perturb_df.to_csv(os.path.join(WS_RESULTS_DIR, "ablation2_random_perturbation.csv"), index=False)

# ---------------------------------------------------------------------------
# 6) Ablation 3 — Component dropout
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("ABLATION 3 — Component dropout (set one weight to 0)")
print("=" * 70)

ws_dropout = [{"dropped": "(none — paper)", "AP": ws_ap_paper,
               f"P@{WS_K}": ws_p_paper, f"R@{WS_K}": ws_r_paper, "ΔAP": 0.0}]
for key in WS_W_PAPER:
    w = {k: (0.0 if k == key else v) for k, v in WS_W_PAPER.items()}
    s = ws_compute_score(ws_df, w)
    ws_dropout.append({
        "dropped": key,
        "AP": ws_ap(ws_y, s),
        f"P@{WS_K}": ws_pk(ws_y, s, WS_K),
        f"R@{WS_K}": ws_rk(ws_y, s, WS_K),
        "ΔAP": ws_ap(ws_y, s) - ws_ap_paper,
    })
ws_dropout_df = pd.DataFrame(ws_dropout)
print(ws_dropout_df.round(3).to_string(index=False))
ws_dropout_df.to_csv(os.path.join(WS_RESULTS_DIR, "ablation3_dropout.csv"), index=False)

# ---------------------------------------------------------------------------
# 7) Plots
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

# (a) λ sweep
ax = axes[0]
lam_x = [0, 0.1, 0.25, 0.5, 1, 2, 4, 10, 50]   # treat ∞ as 50 for plotting
ap_y  = ws_sweep_df["AP"].values
p_y   = ws_sweep_df[f"P@{WS_K}"].values
ax.plot(lam_x, ap_y, "o-", color="C0", label="AP")
ax.plot(lam_x, p_y,  "s-", color="C1", label=f"P@{WS_K}")
ax.axvline(1.0, ls="--", c="gray", alpha=0.7, label="Paper (λ = 1)")
ax.set_xscale("symlog", linthresh=0.1)
ax.set_xticks(lam_x)
ax.set_xticklabels(["0", "0.1", "0.25", "0.5", "1", "2", "4", "10", "∞"])
ax.set_xlabel("λ_LLM (LLM-block scaling factor)")
ax.set_ylabel("Score")
ax.set_title("LLM-vs-lexical balance sweep")
ax.legend(); ax.grid(alpha=0.3)
ax.set_ylim(0, 1)

# (b) Random perturbation histogram
ax = axes[1]
ax.hist(ws_perturb_df["AP"], bins=25, color="C0", alpha=0.65, edgecolor="white")
ax.axvline(ws_ap_paper, ls="--", c="red", lw=2,
           label=f"Paper AP = {ws_ap_paper:.3f}")
ax.axvline(ws_perturb_df["AP"].mean(), ls=":", c="black",
           label=f"Mean = {ws_perturb_df['AP'].mean():.3f}")
ax.set_xlabel("AP")
ax.set_ylabel("Count")
ax.set_title(f"Random perturbation  (×0.5–×1.5 on each weight, N = {WS_N_PERTURB})")
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(WS_RESULTS_DIR, "weight_sensitivity.pdf"), bbox_inches="tight")
fig.savefig(os.path.join(WS_RESULTS_DIR, "weight_sensitivity.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"\n✓ All outputs saved to {WS_RESULTS_DIR}")